# 03 LU 与 PLU 分解

高斯消元不仅可以求解一个右端项，还可以被保存为矩阵分解。LU 分解把系数矩阵写成下三角矩阵与上三角矩阵的乘积；带主元时写成 $PA=LU$。


In [ ]:
import pathlib
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch06_direct_linear_systems"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import lu_doolittle, lu_solve, plu_factorization
from py_sc.direct_linear import relative_residual


## 从 $A=LU$ 推导 Doolittle 公式

Doolittle 分解约定 $L$ 的主对角元为 1。由

$$
A=LU
$$

的第 $(i,j)$ 个元素关系

$$
a_{ij}=\sum_{k=1}^{n}l_{ik}u_{kj}
$$

可得递推公式：

$$
u_{ij}=a_{ij}-\sum_{k=1}^{i-1}l_{ik}u_{kj},
$$

$$
l_{ji}=\frac{a_{ji}-\sum_{k=1}^{i-1}l_{jk}u_{ki}}{u_{ii}}.
$$


In [ ]:
def teaching_lu_doolittle(A):
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    L = np.eye(n)
    U = np.zeros_like(A)
    for i in range(n):
        for j in range(i, n):
            U[i, j] = A[i, j] - L[i, :i] @ U[:i, j]
        for j in range(i + 1, n):
            L[j, i] = (A[j, i] - L[j, :i] @ U[:i, i]) / U[i, i]
    return L, U

A = np.array([[4.0, 2.0, 1.0], [2.0, 5.0, 3.0], [1.0, 3.0, 6.0]])
L, U = teaching_lu_doolittle(A)
print("L=\n", L)
print("U=\n", U)
print("重构误差:", np.linalg.norm(A - L @ U))


## 用 LU 求解多个右端项

分解完成后，对每个右端项只需

$$
L\boldsymbol{y}=\boldsymbol{b},
\qquad
U\boldsymbol{x}=\boldsymbol{y}.
$$

因此多个右端项时，LU 的优势很明显：分解一次，重复前代和回代。


In [ ]:
B = np.array([[1.0, 0.0], [2.0, 1.0], [3.0, -1.0]])
formal_L, formal_U = lu_doolittle(A)
X = lu_solve(formal_L, formal_U, B)
print("正式实现重构误差:", np.linalg.norm(A - formal_L @ formal_U))
print("多个右端项残差:", np.linalg.norm(B - A @ X) / np.linalg.norm(B))


## 置换矩阵和 $PA=LU$

若需要行交换，分解写成

$$
PA=LU.
$$

其中 $P$ 是置换矩阵。实际程序中通常不用显式矩阵 $P$，而是用置换向量记录行顺序。若置换向量为 `p`，则 `A[p, :]` 表示 $PA$。


In [ ]:
B_matrix = np.array([[0.0, 2.0, 1.0], [2.0, 2.0, 3.0], [4.0, -1.0, 2.0]])
b = np.array([1.0, 2.0, 3.0])
plu = plu_factorization(B_matrix)
print("置换向量:", plu.permutation)
print("PA-LU 的范数:", np.linalg.norm(B_matrix[plu.permutation, :] - plu.L @ plu.U))
x = lu_solve(plu.L, plu.U, b, permutation=plu.permutation)
print("解:", x)
print("残差:", relative_residual(B_matrix, x, b))


## 不选主元方法的局限

不选主元 Doolittle 分解要求前导主元不为零。下面矩阵可逆，但第一步主元为零，必须进行行交换。


In [ ]:
try:
    lu_doolittle(B_matrix)
except ValueError as exc:
    print("不选主元 LU 失败:", exc)
print("PLU 可以继续，条件数:", np.linalg.cond(B_matrix))


## 小结

* LU 分解保存了高斯消元的结果。
* Doolittle 约定 $L$ 的对角元为 1。
* 带部分选主元时应理解 $PA=LU$，程序中通常使用置换向量而不是显式置换矩阵。
* LU/PLU 适合多个右端项，因为分解成本只支付一次。
